# Train Variable-Width Base-10 Addition MLP

This notebook trains a reusable variable-width MLP on factual data from a two-digit base-10 addition SCM.

- Inputs: `A1,B1,A2,B2` where each digit is a 10-d one-hot vector
- Model input width: `4 * 10 = 40`
- Output classes: integers `0..198` (`99 + 99 = 198`)

The saved checkpoint is loaded by both `addition_gw.ipynb` and `addition_das.ipynb`.


In [ ]:
import os
import random
import itertools
from typing import Dict, List

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

from pyvene import CausalModel

from variable_width_mlp import VariableWidthMLPConfig, VariableWidthMLPForClassification


## Configuration


In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Training sizes.
N_FACTUAL_TRAIN = 120000
N_FACTUAL_TEST = 12000

# Model/training hyperparameters.
HIDDEN_DIMS = [128, 128]
INPUT_DIM = 40
NUM_CLASSES = 200
DROPOUT = 0.0
ACTIVATION = "relu"
LEARNING_RATE = 2e-3
TRAIN_EPOCHS = 40
TRAIN_BATCH_SIZE = 1024
EVAL_BATCH_SIZE = 512

# Output checkpoint path consumed by GW and DAS notebooks.
MODEL_CHECKPOINT_PATH = "artifacts/addition_mlp.pt"


## SCM Definition (Base-10, Low-Digit-First)

Semantics are locked to low-digit-first carry flow:

- `S1, C1 = add(A1, B1)` (ones place)
- `S2, C2 = add(A2, B2, C1)` (tens place plus incoming carry)
- `O = 100*C2 + 10*S2 + S1` in `{0,...,198}`


In [ ]:
one_hot_digits = [np.eye(10, dtype=np.float32)[d] for d in range(10)]


def as_digit(x):
    arr = np.array(x).reshape(-1)
    if arr.size == 1:
        return int(arr[0])
    return int(arr.argmax())


variables = ["A1", "B1", "A2", "B2", "S1", "C1", "T2", "S2", "C2", "O"]

values = {
    "A1": one_hot_digits,
    "B1": one_hot_digits,
    "A2": one_hot_digits,
    "B2": one_hot_digits,
    "S1": list(range(10)),
    "C1": [0, 1],
    "T2": list(range(19)),
    "S2": list(range(10)),
    "C2": [0, 1],
    "O": list(range(199)),
}

parents = {
    "A1": [],
    "B1": [],
    "A2": [],
    "B2": [],
    "S1": ["A1", "B1"],
    "C1": ["A1", "B1"],
    "T2": ["A2", "B2"],
    "S2": ["T2", "C1"],
    "C2": ["T2", "C1"],
    "O": ["C2", "S2", "S1"],
}


def FILLER():
    return one_hot_digits[0]


functions = {
    "A1": FILLER,
    "B1": FILLER,
    "A2": FILLER,
    "B2": FILLER,
    "S1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) % 10,
    "C1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) // 10,
    "T2": lambda a2, b2: as_digit(a2) + as_digit(b2),
    "S2": lambda t2, c1: (int(t2) + int(c1)) % 10,
    "C2": lambda t2, c1: (int(t2) + int(c1)) // 10,
    "O": lambda c2, s2, s1: int(100 * int(c2) + 10 * int(s2) + int(s1)),
}

addition_model = CausalModel(variables, values, parents, functions)


In [ ]:
addition_model.print_structure()

In [ ]:
# Truth-table sanity checks over all 10,000 input combinations.

def assignment_from_digits(digits):
    a1, b1, a2, b2 = [int(x) for x in digits]
    return {
        "A1": one_hot_digits[a1],
        "B1": one_hot_digits[b1],
        "A2": one_hot_digits[a2],
        "B2": one_hot_digits[b2],
    }


for digits in itertools.product(range(10), repeat=4):
    setting = addition_model.run_forward(assignment_from_digits(digits))
    a1, b1, a2, b2 = digits

    s1 = (a1 + b1) % 10
    c1 = (a1 + b1) // 10
    s2 = (a2 + b2 + c1) % 10
    c2 = (a2 + b2 + c1) // 10
    o = 100 * c2 + 10 * s2 + s1

    assert int(setting["S1"]) == s1
    assert int(setting["C1"]) == c1
    assert int(setting["S2"]) == s2
    assert int(setting["C2"]) == c2
    assert int(setting["O"]) == o

print("SCM checks passed for all 10,000 assignments.")


## Variable-Width MLP


In [ ]:
from variable_width_mlp import VariableWidthMLPConfig, VariableWidthMLPForClassification

print("Using shared MLP definitions from variable_width_mlp.py")


## Factual Dataset and Training


In [ ]:
all_assignments = [
    assignment_from_digits(digits)
    for digits in itertools.product(range(10), repeat=4)
]


def factual_input_sampler():
    return random.choice(all_assignments)


train_examples = addition_model.generate_factual_dataset(N_FACTUAL_TRAIN, factual_input_sampler)
X_train = torch.stack([ex["input_ids"] for ex in train_examples]).to(torch.float32)
y_train = torch.stack([ex["labels"] for ex in train_examples]).to(torch.long)

print("X_train shape:", tuple(X_train.shape))  # [N, 40]
print("y_train shape:", tuple(y_train.shape))  # [N]
print("y range:", int(y_train.min()), int(y_train.max()))

cfg = VariableWidthMLPConfig(
    input_dim=INPUT_DIM,
    hidden_dims=HIDDEN_DIMS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
    activation=ACTIVATION,
)

model = VariableWidthMLPForClassification(cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=TRAIN_BATCH_SIZE, shuffle=True)

model.train()
for epoch in range(TRAIN_EPOCHS):
    running_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).squeeze(-1)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs_embeds=xb.unsqueeze(1))[0]
        # print(logits, logits.shape, yb, yb.shape)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.detach().cpu()) * xb.shape[0]

    if (epoch + 1) % max(1, TRAIN_EPOCHS // 10) == 0:
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch + 1:>3}/{TRAIN_EPOCHS}: loss={epoch_loss:.6f}")


In [ ]:
# Quick factual evaluation.
model.eval()

test_examples = addition_model.generate_factual_dataset(N_FACTUAL_TEST, factual_input_sampler)
X_test = torch.stack([ex["input_ids"] for ex in test_examples]).to(torch.float32)
y_test = torch.tensor([int(ex["labels"].item()) for ex in test_examples], dtype=torch.long)

pred_chunks = []
with torch.no_grad():
    for start in range(0, len(X_test), EVAL_BATCH_SIZE):
        end = min(start + EVAL_BATCH_SIZE, len(X_test))
        logits = model(inputs_embeds=X_test[start:end].to(device).unsqueeze(1))[0]
        pred_chunks.append(logits.detach().cpu())

pred_logits = torch.cat(pred_chunks, dim=0)
pred_labels = pred_logits.argmax(1)
acc = accuracy_score(y_test.numpy(), pred_labels.numpy())

print(f"Factual test accuracy: {acc:.4f}")
print("Example predictions (pred -> gold):")
for i in range(8):
    print(int(pred_labels[i].item()), "->", int(y_test[i].item()))


## Save Reusable Checkpoint


In [ ]:
os.makedirs(os.path.dirname(MODEL_CHECKPOINT_PATH), exist_ok=True)

payload = {
    "model_state_dict": model.state_dict(),
    "model_config": cfg.to_dict(),
    "metadata": {
        "seed": SEED,
        "task": "two_digit_base10_addition_onehot",
        "scm_variables": ["A1", "B1", "A2", "B2", "S1", "C1", "S2", "C2"],
        "target": "O",
        "output_classes": list(range(NUM_CLASSES)),
    },
}

torch.save(payload, MODEL_CHECKPOINT_PATH)
print("Saved checkpoint:", MODEL_CHECKPOINT_PATH)
print("Saved hidden dims:", cfg.hidden_dims)


## Load Snippet for Other Notebooks

```python
from variable_width_mlp import VariableWidthMLPConfig, VariableWidthMLPForClassification

checkpoint = torch.load("artifacts/addition_variable_width_mlp.pt", map_location=device)
cfg = VariableWidthMLPConfig(**checkpoint["model_config"])
trained = VariableWidthMLPForClassification(cfg).to(device)
trained.load_state_dict(checkpoint["model_state_dict"])
trained.eval()
```
